# PHASE 2 · UNIFIED RISK MODELS

<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>01 · Environment and dependencies</b><br/>Install dependencies, configure paths, and check the GPU.</div>

In [ ]:
%pip -q install "tabpfn==9.1.0" "catboost>=1.2.8" "shap>=0.46" "pyarrow>=15" "scikit-learn>=1.5" "joblib>=1.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 27.9 MB/s eta 0:00:00


In [ ]:
import os, json, sys, gc, hashlib, importlib.metadata, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import shap
from catboost import CatBoostClassifier
from tabpfn import TabPFNClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,
                             recall_score, f1_score, confusion_matrix, brier_score_loss)
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: expecting an equivalent mounted Drive directory.')

SEED = 42
ROOT = Path('/content/drive/MyDrive/GC1/Temporal_snapshots')
FEATURE_DIR = ROOT / 'feature_engineering_output'
PRIOR_DIR = ROOT / 'model_training_output'
OUT = ROOT / 'model_training_efficient_investigation'
for folder in ['models', 'metrics', 'predictions', 'shap', 'triggers', 'audits']:
    (OUT / folder).mkdir(parents=True, exist_ok=True)
DATA_PATH = FEATURE_DIR / 'tabpfn_labelled_dataset.parquet'
FEATURE_PATH = FEATURE_DIR / 'selected_feature_list.csv'
SPLIT_PATH = PRIOR_DIR / 'project_split_manifest.csv'
for p in [DATA_PATH, FEATURE_PATH, SPLIT_PATH]:
    if not p.is_file():
        raise FileNotFoundError(f'Required upstream artifact is missing: {p}')
if not torch.cuda.is_available():
    raise RuntimeError('TabPFN-2.5 requires a supported GPU for this notebook. Enable T4 GPU in Colab.')
DEVICE = 'cuda'
print('GPU:', torch.cuda.get_device_name(0))
print('TabPFN:', importlib.metadata.version('tabpfn'))
print('CatBoost:', importlib.metadata.version('catboost'))
print('Output (new folder only):', OUT)

Mounted at /content/drive
GPU: Tesla T4
TabPFN: 9.1.0
CatBoost: 1.2.10
Output (new folder only): /content/drive/MyDrive/GC1/Temporal_snapshots/model_training_efficient_investigation


<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>02 · Load finalized Notebook 2 outputs</b><br/>Reuse the exact labelled feature matrix and the existing project split. Do not regenerate labels.</div>

In [ ]:
TARGETS = ['schedule_risk_label', 'scope_risk_label', 'dependency_risk_label']
KEYS = ['instance_id', 'week_number', 'checkpoint_time']
dataset = pd.read_parquet(DATA_PATH)
feature_list = pd.read_csv(FEATURE_PATH)
assert 'feature_name' in feature_list.columns
FEATURES = feature_list['feature_name'].astype(str).tolist()
assert len(FEATURES) == 34 and len(FEATURES) == len(set(FEATURES))
assert set(KEYS + TARGETS + FEATURES).issubset(dataset.columns)
dataset['checkpoint_time'] = pd.to_datetime(dataset['checkpoint_time'], utc=True, format='mixed')
manifest = pd.read_csv(SPLIT_PATH)
assert {'instance_id','partition'}.issubset(manifest.columns)
manifest['partition'] = manifest['partition'].astype(str).str.lower().replace({'val':'validation'})
assert set(manifest['partition']) == {'train','validation','test'}
assert not manifest['instance_id'].duplicated().any()
assert set(manifest['instance_id']) == set(dataset['instance_id'])
assert not dataset.duplicated(KEYS).any()
assert dataset[TARGETS].isin([0,1]).all().all()
assert np.isfinite(dataset[FEATURES].to_numpy(dtype=float)).all()
dataset = dataset.merge(manifest[['instance_id','partition']], on='instance_id', validate='many_to_one')
dataset = dataset.sort_values(['instance_id','checkpoint_time']).reset_index(drop=True)
train = dataset.loc[dataset.partition == 'train'].copy().reset_index(drop=True)
val = dataset.loc[dataset.partition == 'validation'].copy().reset_index(drop=True)
test = dataset.loc[dataset.partition == 'test'].copy().reset_index(drop=True)
assert (len(dataset),len(train),len(val),len(test)) == (987,592,198,197)
assert (train.instance_id.nunique(),val.instance_id.nunique(),test.instance_id.nunique()) == (18,6,6)
assert not (set(train.instance_id)&set(val.instance_id) or set(train.instance_id)&set(test.instance_id) or set(val.instance_id)&set(test.instance_id))
print('Loaded:',len(dataset),'snapshots /',dataset.instance_id.nunique(),'projects /',len(FEATURES),'features')
display(dataset.groupby('partition')[TARGETS].agg(['sum','count']))

Loaded: 987 snapshots / 30 projects / 34 features


schedule_risk_label       scope_risk_label        \
                           sum count              sum count   
partition                                                     
test                        52   197               37   197   
train                      161   592               97   592   
validation                  52   198               33   198   

           dependency_risk_label        
                             sum count  
partition                               
test                          14   197  
train                         58   592  
validation                    16   198

<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>03 · Freeze heterogeneous model configuration</b><br/>Keep the existing 34-feature contract. Schedule: TabPFN-2.5. Scope and Dependency: CatBoost. No decision-risk model.</div>

In [ ]:
MODEL_SPEC = {
    'schedule_risk_label': {'algorithm':'TabPFN-2.5', 'features':FEATURES.copy(), 'reliability':'EXPERIMENTAL'},
    'scope_risk_label': {'algorithm':'CatBoost', 'features':FEATURES.copy(), 'reliability':'LOW_RELIABILITY'},
    'dependency_risk_label': {'algorithm':'CatBoost', 'features':FEATURES.copy(), 'reliability':'LOW_RELIABILITY'}
}
CAT_PARAMS = dict(iterations=250, depth=3, learning_rate=0.035, l2_leaf_reg=8,
                  loss_function='Logloss', verbose=False, random_seed=SEED,
                  thread_count=2, allow_writing_files=False)
TABPFN_ESTIMATORS = 4
# The pretrained v2.5 checkpoint is resolved explicitly, avoiding a silent default-version switch.
from tabpfn.model_loading import resolve_model_path
try:
    from google.colab import userdata
    token = userdata.get('TABPFN_TOKEN')
    if token:
        os.environ['TABPFN_TOKEN'] = token
except (ImportError, KeyError, AttributeError):
    pass
try:
    model_paths, _, model_names, model_type = resolve_model_path(
        model_path=None, which='classifier', version='v2.5')
    assert model_paths and model_type == 'classifier'
    TABPFN_PATH = Path(model_paths[0])
except Exception as e:
    raise RuntimeError('Cannot resolve TabPFN v2.5 checkpoint. Accept its license and configure TABPFN_TOKEN if needed.') from e
print('TabPFN v2.5 checkpoint:', TABPFN_PATH)
print('Configured models:')
display(pd.DataFrame([{'risk':k,'model':v['algorithm'],'features':len(v['features']),'reliability':v['reliability']} for k,v in MODEL_SPEC.items()]))

TabPFN v2.5 checkpoint: /root/.cache/tabpfn/tabpfn-v2.5-classifier-v2.5_default.ckpt
Configured models:


,risk,model,features,reliability
0,schedule_risk_label,TabPFN-2.5,34,EXPERIMENTAL
1,scope_risk_label,CatBoost,34,LOW_RELIABILITY
2,dependency_risk_label,CatBoost,34,LOW_RELIABILITY


<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>04 · Reproducible training-only cross-validation</b><br/>Use grouped folds; never mix checkpoints from the same project between a fold's training and validation partitions.</div>

In [ ]:
project_stats = train.groupby('instance_id')[TARGETS].sum()
positive_dependency_projects = project_stats.index[project_stats.dependency_risk_label > 0].tolist()
assert len(positive_dependency_projects) == 4
fold_projects = [[p] for p in positive_dependency_projects]
other_projects = sorted(set(train.instance_id) - set(positive_dependency_projects))
np.random.default_rng(SEED).shuffle(other_projects)
for project in other_projects:
    def fold_score(k):
        z = project_stats.loc[fold_projects[k] + [project]]
        return len(z)*10 + z.schedule_risk_label.sum()/10 + z.scope_risk_label.sum()/10
    fold_projects[min(range(4),key=fold_score)].append(project)
FOLDS=[]; fold_report=[]
for fold_number,projects in enumerate(fold_projects,1):
    mask=train.instance_id.isin(projects).to_numpy()
    tr_idx=np.flatnonzero(~mask);va_idx=np.flatnonzero(mask)
    assert set(train.iloc[tr_idx].instance_id).isdisjoint(set(train.iloc[va_idx].instance_id))
    for t in TARGETS:
        assert train.iloc[tr_idx][t].nunique()==2 and train.iloc[va_idx][t].nunique()==2
    FOLDS.append((tr_idx,va_idx))
    fold_report.append({'fold':fold_number,'projects':len(projects),'rows':len(va_idx),**{t+'_positives':int(train.iloc[va_idx][t].sum()) for t in TARGETS}})
assert sorted(np.concatenate([v for _,v in FOLDS]).tolist()) == list(range(len(train)))
fold_report_df=pd.DataFrame(fold_report)
display(fold_report_df)
fold_report_df.to_csv(OUT/'audits'/'training_fold_summary.csv',index=False)
print('PASS: four project-grouped CV folds')

,fold,projects,rows,schedule_risk_label_positives,scope_risk_label_positives,dependency_risk_label_positives
0,1,4,132,63,32,24
1,2,5,165,15,17,2
2,3,4,129,31,28,4
3,4,5,166,52,20,28


PASS: four project-grouped CV folds


In [ ]:
def make_model(target, y_train):
    spec=MODEL_SPEC[target]
    if spec['algorithm']=='TabPFN-2.5':
        return TabPFNClassifier(model_path=TABPFN_PATH,device=DEVICE,
                                n_estimators=TABPFN_ESTIMATORS,random_state=SEED)
    y=np.asarray(y_train,dtype=int)
    positive=int(y.sum());negative=len(y)-positive
    assert positive>0 and negative>0
    return CatBoostClassifier(**CAT_PARAMS,class_weights=[1.0,negative/positive])

def predict_positive(model, X):
    classes=list(model.classes_)
    assert 1 in classes
    p=np.asarray(model.predict_proba(X))[:,classes.index(1)]
    assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
    return p

cv_parts=[]
for target in TARGETS:
    cols=MODEL_SPEC[target]['features']
    for fold_number,(tr_idx,va_idx) in enumerate(FOLDS,1):
        tr=train.iloc[tr_idx];va=train.iloc[va_idx]
        model=make_model(target,tr[target])
        model.fit(tr[cols],tr[target].astype(int))
        p=predict_positive(model,va[cols])
        result=va[KEYS].copy()
        result['risk']=target;result['fold']=fold_number
        result['actual_label']=va[target].to_numpy(dtype=int)
        result['probability']=p
        cv_parts.append(result)
        print(target,'fold',fold_number,'AP:',round(average_precision_score(va[target],p),4))
        del model
        gc.collect()
        torch.cuda.empty_cache()
cv_predictions=pd.concat(cv_parts,ignore_index=True)
assert len(cv_predictions)==592*3 and not cv_predictions.duplicated(KEYS+['risk']).any()
cv_predictions.to_parquet(OUT/'predictions'/'training_only_cv_predictions.parquet',index=False)
print('PASS: all 12 grouped-CV model fits')

tabpfn-v2.5-classifier-v2.5_default.ckpt: reconstructing file:   0%|          |  0.00B / 42.9MB            

tabpfn-v2.5-classifier-v2.5_default.ckpt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

schedule_risk_label fold 1 AP: 0.5129
schedule_risk_label fold 2 AP: 0.1648
schedule_risk_label fold 3 AP: 0.3355
schedule_risk_label fold 4 AP: 0.5943
scope_risk_label fold 1 AP: 0.2389
scope_risk_label fold 2 AP: 0.4349
scope_risk_label fold 3 AP: 0.3859
scope_risk_label fold 4 AP: 0.1404
dependency_risk_label fold 1 AP: 0.7797
dependency_risk_label fold 2 AP: 0.0119
dependency_risk_label fold 3 AP: 0.0322
dependency_risk_label fold 4 AP: 0.8963
PASS: all 12 grouped-CV model fits


In [ ]:
def rank_metrics(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    return {'records':len(y),'positives':int(y.sum()),'prevalence':float(y.mean()),
            'PR_AUC':float(average_precision_score(y,p)) if y.sum()>0 else np.nan,
            'ROC_AUC':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan}
cv_metrics=pd.DataFrame([{'risk':r,**rank_metrics(g.actual_label,g.probability)} for r,g in cv_predictions.groupby('risk')])
cv_fold_metrics=pd.DataFrame([{'risk':r,'fold':f,**rank_metrics(g.actual_label,g.probability)} for (r,f),g in cv_predictions.groupby(['risk','fold'])])
display(cv_metrics.round(4));display(cv_fold_metrics.round(4))
cv_metrics.to_csv(OUT/'metrics'/'training_cv_pooled.csv',index=False)
cv_fold_metrics.to_csv(OUT/'metrics'/'training_cv_folds.csv',index=False)

,risk,records,positives,prevalence,PR_AUC,ROC_AUC
0,dependency_risk_label,592,58,0.0980,0.5692,0.8482
1,schedule_risk_label,592,161,0.2720,0.3309,0.5523
2,scope_risk_label,592,97,0.1639,0.2585,0.6599


,risk,fold,records,positives,prevalence,PR_AUC,ROC_AUC
0,dependency_risk_label,1,132,24,0.1818,0.7797,0.9626
1,dependency_risk_label,2,165,2,0.0121,0.0119,0.2515
2,dependency_risk_label,3,129,4,0.0310,0.0322,0.3560
3,dependency_risk_label,4,166,28,0.1687,0.8963,0.9741
4,schedule_risk_label,1,132,63,0.4773,0.5129,0.5862
5,schedule_risk_label,2,165,15,0.0909,0.1648,0.6364
6,schedule_risk_label,3,129,31,0.2403,0.3355,0.5655
7,schedule_risk_label,4,166,52,0.3133,0.5943,0.7115
8,scope_risk_label,1,132,32,0.2424,0.2389,0.5041
9,scope_risk_label,2,165,17,0.1030,0.4349,0.8168


<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>05 · Fit three final models on training projects</b><br/>CatBoost trains trees on the labelled data; TabPFN conditions its pretrained transformer on the training context.</div>

In [ ]:
MODELS={};VAL_PROBS={}
for target in TARGETS:
    cols=MODEL_SPEC[target]['features']
    model=make_model(target,train[target])
    model.fit(train[cols],train[target].astype(int))
    MODELS[target]=model
    VAL_PROBS[target]=predict_positive(model,val[cols])
    print('Fitted:',target,MODEL_SPEC[target]['algorithm'],'validation AP',round(average_precision_score(val[target],VAL_PROBS[target]),4))
    if MODEL_SPEC[target]['algorithm']=='CatBoost':
        model.save_model(str(OUT/'models'/f'{target}.cbm'))
    else:
        context=train[cols].reset_index(drop=True).copy()
        context['target_label']=train[target].to_numpy(dtype=int)
        context.to_parquet(OUT/'models'/f'{target}_training_context.parquet',index=False)
        # This context + the versioned v2.5 checkpoint is the supported reproducible fallback.
        try:
            import joblib
            joblib.dump(model,OUT/'models'/f'{target}_tabpfn.joblib',compress=3)
            print('TabPFN fitted object serialized')
        except Exception as exc:
            print('Fitted-object serialization unavailable; saved training context remains usable:',type(exc).__name__)
assert len(MODELS)==3
print('PASS: three final models fitted and saved/reconstructable')

Fitted: schedule_risk_label TabPFN-2.5 validation AP 0.3142
TabPFN fitted object serialized
Fitted: scope_risk_label CatBoost validation AP 0.3806
Fitted: dependency_risk_label CatBoost validation AP 0.1668
PASS: three final models fitted and saved/reconstructable


<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>06 · Select investigation thresholds using validation only</b><br/>F1-based selection is an operational choice, not probability calibration. Thresholds are risk-specific and frozen before test inference.</div>

In [ ]:
def choose_threshold(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    assert len(y)==len(p) and set(np.unique(y))=={0,1}
    candidates=[]
    for threshold in np.unique(p):
        pred=(p>=threshold).astype(int)
        candidates.append((f1_score(y,pred,zero_division=0),
                           precision_score(y,pred,zero_division=0),float(threshold)))
    # maximize F1, then precision, then prefer the higher threshold
    return float(max(candidates)[2])
THRESHOLDS={t:choose_threshold(val[t],VAL_PROBS[t]) for t in TARGETS}
validation_rows=[]
for t in TARGETS:
    y=val[t].to_numpy(dtype=int);p=VAL_PROBS[t];pred=(p>=THRESHOLDS[t]).astype(int)
    validation_rows.append({'risk':t,'algorithm':MODEL_SPEC[t]['algorithm'],'threshold':THRESHOLDS[t],
        **rank_metrics(y,p),'precision':precision_score(y,pred,zero_division=0),
        'recall':recall_score(y,pred,zero_division=0),'F1':f1_score(y,pred,zero_division=0),
        'predicted_positives':int(pred.sum())})
validation_metrics=pd.DataFrame(validation_rows)
display(validation_metrics.round(5))
validation_metrics.to_csv(OUT/'metrics'/'validation_metrics_thresholds.csv',index=False)
assert set(THRESHOLDS)==set(TARGETS)
print('PASS: thresholds frozen using validation only')

,risk,algorithm,threshold,records,positives,prevalence,PR_AUC,ROC_AUC,precision,recall,F1,predicted_positives
0,schedule_risk_label,TabPFN-2.5,0.28241,198,52,0.26263,0.31421,0.58917,0.34579,0.71154,0.46541,107
1,scope_risk_label,CatBoost,0.14764,198,33,0.16667,0.38059,0.74858,0.33846,0.66667,0.44898,65
2,dependency_risk_label,CatBoost,0.12626,198,16,0.08081,0.16680,0.62225,0.11905,0.62500,0.20000,84


PASS: thresholds frozen using validation only


<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>07 · Exploratory evaluation on previously inspected test projects</b><br/>Never adjust the model or thresholds based on these results. They do not constitute fresh unbiased external validation.</div>

In [ ]:
def predict_partition(frame, name):
    pieces=[]
    for target in TARGETS:
        cols=MODEL_SPEC[target]['features']
        p=predict_positive(MODELS[target],frame[cols])
        part=frame[KEYS].copy()
        part['partition']=name;part['risk']=target
        part['algorithm']=MODEL_SPEC[target]['algorithm']
        part['actual_label']=frame[target].to_numpy(dtype=int)
        part['probability']=p
        part['threshold']=THRESHOLDS[target]
        part['predicted_label']=(p>=THRESHOLDS[target]).astype(int)
        pieces.append(part)
    return pd.concat(pieces,ignore_index=True)
validation_predictions=predict_partition(val,'validation')
test_predictions=predict_partition(test,'test')
assert len(validation_predictions)==198*3 and len(test_predictions)==197*3
validation_predictions.to_parquet(OUT/'predictions'/'validation_predictions.parquet',index=False)
test_predictions.to_parquet(OUT/'predictions'/'exploratory_test_predictions.parquet',index=False)
print('Validation and exploratory test predictions generated')

Validation and exploratory test predictions generated


In [ ]:
def evaluate_partition(predictions):
    records=[]
    for risk,g in predictions.groupby('risk'):
        y=g.actual_label.to_numpy(dtype=int);p=g.probability.to_numpy(dtype=float)
        pred=g.predicted_label.to_numpy(dtype=int)
        tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
        records.append({'risk':risk,**rank_metrics(y,p),
            'precision':float(precision_score(y,pred,zero_division=0)),
            'recall':float(recall_score(y,pred,zero_division=0)),
            'F1':float(f1_score(y,pred,zero_division=0)),
            'Brier':float(brier_score_loss(y,p)),
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)})
    return pd.DataFrame(records)
test_metrics=evaluate_partition(test_predictions)
project_metrics=pd.DataFrame([
    {'risk':risk,'instance_id':project,**rank_metrics(g.actual_label,g.probability),
     'precision':float(precision_score(g.actual_label,g.predicted_label,zero_division=0)),
     'recall':float(recall_score(g.actual_label,g.predicted_label,zero_division=0)),
     'F1':float(f1_score(g.actual_label,g.predicted_label,zero_division=0))}
    for (risk,project),g in test_predictions.groupby(['risk','instance_id'])])
print('EXPLORATORY HELD-OUT METRICS');display(test_metrics.round(4))
print('PROJECT-LEVEL METRICS');display(project_metrics.round(4))
test_metrics.to_csv(OUT/'metrics'/'exploratory_test_metrics.csv',index=False)
project_metrics.to_csv(OUT/'metrics'/'exploratory_project_metrics.csv',index=False)
assert len(test_metrics)==3 and len(project_metrics)==18

EXPLORATORY HELD-OUT METRICS


,risk,records,positives,prevalence,PR_AUC,ROC_AUC,precision,recall,F1,Brier,TN,FP,FN,TP
0,dependency_risk_label,197,14,0.0711,0.1305,0.7521,0.1525,0.6429,0.2466,0.1075,133,50,5,9
1,schedule_risk_label,197,52,0.2640,0.5176,0.8095,0.4842,0.8846,0.6259,0.1533,96,49,6,46
2,scope_risk_label,197,37,0.1878,0.2972,0.7199,0.2982,0.9189,0.4503,0.1526,80,80,3,34


PROJECT-LEVEL METRICS


,risk,instance_id,records,positives,prevalence,PR_AUC,ROC_AUC,precision,recall,F1
0,dependency_risk_label,AMBARI-I01,33,2,0.0606,0.0464,0.0161,0.0606,1.0000,0.1143
1,dependency_risk_label,ENTESB-I01,33,12,0.3636,0.2489,0.1706,0.2800,0.5833,0.3784
2,dependency_risk_label,JRACLOUD-I01,33,0,0.0000,NaN,NaN,0.0000,0.0000,0.0000
3,dependency_risk_label,MNG-I01,32,0,0.0000,NaN,NaN,0.0000,0.0000,0.0000
4,dependency_risk_label,RUBY-I01,33,0,0.0000,NaN,NaN,0.0000,0.0000,0.0000
5,dependency_risk_label,WT-I01,33,0,0.0000,NaN,NaN,0.0000,0.0000,0.0000
6,schedule_risk_label,AMBARI-I01,33,19,0.5758,0.6192,0.5526,0.5758,1.0000,0.7308
7,schedule_risk_label,ENTESB-I01,33,2,0.0606,0.1051,0.5968,0.0769,0.5000,0.1333
8,schedule_risk_label,JRACLOUD-I01,33,0,0.0000,NaN,NaN,0.0000,0.0000,0.0000
9,schedule_risk_label,MNG-I01,32,2,0.0625,0.0553,0.1667,0.0000,0.0000,0.0000


<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>08 · SHAP: model-compatible local explanations</b><br/>TabPFN uses permutation SHAP (probability units). CatBoost uses TreeSHAP (log-odds units). Do not compare magnitudes across these two units. Explain validation examples and provide an on-demand function for Agentic GraphRAG.</div>

In [ ]:
# Small background ensures the TabPFN permutation explainer remains practical on T4.
SHAP_BACKGROUND = 10
SHAP_TOP_K = 5

def explain_snapshot(row, target):
    """Explain one feature row. Returns a structured, model-consistent SHAP payload."""
    assert target in MODEL_SPEC
    features=MODEL_SPEC[target]['features']
    model=MODELS[target]
    X=pd.DataFrame([row[features].to_numpy(dtype=float)],columns=features)
    if MODEL_SPEC[target]['algorithm']=='CatBoost':
        exp=shap.TreeExplainer(model)
        values=np.asarray(exp.shap_values(X))
        if values.ndim==3:
            values=values[:,:,1]
        sv=np.asarray(values[0],dtype=float)
        base=np.asarray(exp.expected_value)
        base=float(base[-1] if base.ndim else base)
        raw=float(np.asarray(model.predict(X,prediction_type='RawFormulaVal')).reshape(-1)[0])
        if not np.isclose(base+sv.sum(),raw,atol=1e-4):
            raise AssertionError('CatBoost TreeSHAP log-odds additivity failed')
        unit='log_odds';reconstructed=raw
    else:
        background=train[features].sample(n=min(SHAP_BACKGROUND,len(train)),random_state=SEED)
        positive_index=list(model.classes_).index(1)
        def predict_fn(array):
            z=pd.DataFrame(np.asarray(array),columns=features)
            return np.asarray(model.predict_proba(z))[:,positive_index]
        explainer=shap.Explainer(predict_fn,shap.maskers.Independent(background,max_samples=len(background)),
                                algorithm='permutation',feature_names=features)
        exp=explainer(X,max_evals=2*len(features)+1)
        sv=np.asarray(exp.values[0],dtype=float)
        base=float(np.asarray(exp.base_values).reshape(-1)[0])
        reconstructed=float(base+sv.sum())
        unit='probability'
    p=float(predict_positive(model,X)[0])
    if unit=='probability' and not np.isclose(p,reconstructed,atol=1e-3):
        raise AssertionError('TabPFN SHAP probability additivity failed')
    if unit=='log_odds':
        logistic=1/(1+np.exp(-np.clip(reconstructed,-700,700)))
        if not np.isclose(p,logistic,atol=1e-5):
            raise AssertionError('CatBoost SHAP log-odds to probability check failed')
    entries=[{'feature':feature,'feature_value':float(value),'shap_value':float(contrib),
              'direction':'INCREASES_PREDICTION' if contrib>0 else 'DECREASES_PREDICTION' if contrib<0 else 'NEUTRAL'}
             for feature,value,contrib in zip(features,X.iloc[0].to_numpy(dtype=float),sv)]
    entries.sort(key=lambda z:abs(z['shap_value']),reverse=True)
    return {'risk':target,'algorithm':MODEL_SPEC[target]['algorithm'],'probability':p,
            'threshold':float(THRESHOLDS[target]),'shap_unit':unit,'base_value':base,
            'reconstructed_model_output':reconstructed,'top_drivers':entries[:SHAP_TOP_K],
            'all_contributions':entries}

# Representative validation cases: one TP, FP, FN where available for each risk.
explanation_records=[]; explanation_summaries=[]
for target in TARGETS:
    sub=validation_predictions[validation_predictions.risk==target]
    patterns=[('TRUE_POSITIVE',1,1),('FALSE_POSITIVE',0,1),('FALSE_NEGATIVE',1,0)]
    for case_type,actual,predicted in patterns:
        matches=sub[(sub.actual_label==actual)&(sub.predicted_label==predicted)]
        if matches.empty:
            continue
        case=matches.iloc[0]
        row=val[(val.instance_id==case.instance_id)&(val.week_number==case.week_number)]
        assert len(row)==1
        explanation=explain_snapshot(row.iloc[0],target)
        discrepancy=abs(explanation['probability']-float(case.probability))
        summary={'instance_id':str(case.instance_id),'week_number':int(case.week_number),
                 'risk':target,'case_type':case_type,'original_probability':float(case.probability),
                 'shap_inference_probability':explanation['probability'],'probability_difference':discrepancy,
                 'threshold':float(case.threshold),'near_threshold':bool(abs(float(case.probability)-float(case.threshold))<=1e-3),
                 'shap_unit':explanation['shap_unit']}
        explanation_summaries.append(summary)
        for rank,driver in enumerate(explanation['all_contributions'],1):
            explanation_records.append({**summary,**driver,'rank':rank})
        print(target,case_type,'SHAP complete; inference delta:',round(discrepancy,6))
        gc.collect();torch.cuda.empty_cache()
shap_summaries=pd.DataFrame(explanation_summaries)
shap_contributions=pd.DataFrame(explanation_records)
assert not shap_summaries.empty and np.isfinite(shap_contributions.shap_value).all()
display(shap_summaries.round(6))
shap_summaries.to_csv(OUT/'shap'/'validation_shap_cases.csv',index=False)
shap_contributions.to_parquet(OUT/'shap'/'validation_shap_contributions.parquet',index=False)
print('PASS: SHAP explanations generated and exported')

schedule_risk_label TRUE_POSITIVE SHAP complete; inference delta: 0.000203
schedule_risk_label FALSE_POSITIVE SHAP complete; inference delta: 0.000109
schedule_risk_label FALSE_NEGATIVE SHAP complete; inference delta: 6.2e-05
scope_risk_label TRUE_POSITIVE SHAP complete; inference delta: 0.0
scope_risk_label FALSE_POSITIVE SHAP complete; inference delta: 0.0
scope_risk_label FALSE_NEGATIVE SHAP complete; inference delta: 0.0
dependency_risk_label TRUE_POSITIVE SHAP complete; inference delta: 0.0
dependency_risk_label FALSE_POSITIVE SHAP complete; inference delta: 0.0
dependency_risk_label FALSE_NEGATIVE SHAP complete; inference delta: 0.0


,instance_id,week_number,risk,case_type,original_probability,shap_inference_probability,probability_difference,threshold,near_threshold,shap_unit
0,AEROGEAR-I01,3,schedule_risk_label,TRUE_POSITIVE,0.303842,0.303639,0.000203,0.282411,False,probability
1,AEROGEAR-I01,1,schedule_risk_label,FALSE_POSITIVE,0.343373,0.343483,0.000109,0.282411,False,probability
2,AEROGEAR-I01,16,schedule_risk_label,FALSE_NEGATIVE,0.276502,0.276440,0.000062,0.282411,False,probability
3,AEROGEAR-I01,5,scope_risk_label,TRUE_POSITIVE,0.230801,0.230801,0.000000,0.147636,False,log_odds
4,AEROGEAR-I01,1,scope_risk_label,FALSE_POSITIVE,0.419655,0.419655,0.000000,0.147636,False,log_odds
5,JBDS-I01,7,scope_risk_label,FALSE_NEGATIVE,0.044327,0.044327,0.000000,0.147636,False,log_odds
6,AEROGEAR-I01,5,dependency_risk_label,TRUE_POSITIVE,0.250373,0.250373,0.000000,0.126258,False,log_odds
7,AEROGEAR-I01,1,dependency_risk_label,FALSE_POSITIVE,0.570956,0.570956,0.000000,0.126258,False,log_odds
8,JBDS-I01,5,dependency_risk_label,FALSE_NEGATIVE,0.030778,0.030778,0.000000,0.126258,False,log_odds


PASS: SHAP explanations generated and exported


<div style="background:#edf4fc;border-left:6px solid #2865a0;padding:17px 20px;border-radius:10px"><span style="font-size:19px;font-weight:750;color:#17385a">09 · Stateful investigation policy</span><div style="font-size:13px;color:#43576d;margin-top:6px">All risk predictions remain stored. Only novel, escalated, evidence-changed or periodically refreshed risks are queued for Agentic GraphRAG.</div></div>

In [ ]:
# ------------------------------------------------------------
# Policy configuration: operational values, NOT learned from the test set
# ------------------------------------------------------------
from dataclasses import dataclass, asdict
from collections import defaultdict

RISK_NAMES = {
    'schedule_risk_label': 'schedule',
    'scope_risk_label': 'scope',
    'dependency_risk_label': 'dependency'
}
EVIDENCE_HINTS = {
    'schedule': ['issue_status_history', 'reopenings', 'blockers', 'milestones', 'recent_events'],
    'scope': ['epic_membership', 'issue_reclassification', 'requirement_changes', 'recent_events'],
    'dependency': ['blocking_links', 'upstream_issues', 'downstream_issues', 'dependency_chains', 'recent_events']
}

@dataclass(frozen=True)
class InvestigationPolicy:
    escalation_delta: float = 0.10  # absolute score increase from last investigated score
    periodic_refresh_weeks: int = 4
    minimum_escalation_gap_weeks: int = 2
    near_threshold_margin: float = 0.001

POLICY = InvestigationPolicy()
assert 0 < POLICY.escalation_delta < 1
assert POLICY.periodic_refresh_weeks >= 1
assert POLICY.minimum_escalation_gap_weeks >= 0
print('Frozen investigation policy:', asdict(POLICY))


Frozen investigation policy: {'escalation_delta': 0.1, 'periodic_refresh_weeks': 4, 'minimum_escalation_gap_weeks': 2, 'near_threshold_margin': 0.001}


<div style="background:#edf4fc;border-left:6px solid #2865a0;padding:17px 20px;border-radius:10px"><span style="font-size:19px;font-weight:750;color:#17385a">10 · Chronological replay engine</span><div style="font-size:13px;color:#43576d;margin-top:6px">Per-project and per-risk memory; distinguishes alert decisions from actual investigation scheduling. Explicit source event IDs are optional and point-in-time only.</div></div>

In [ ]:
def _as_utc(value):
    stamp = pd.Timestamp(value)
    if stamp.tzinfo is None:
        stamp = stamp.tz_localize('UTC')
    else:
        stamp = stamp.tz_convert('UTC')
    return stamp


def _safe_event_ids(event_ids_by_checkpoint, project, week, risk):
    """Optional caller-supplied IDs observed by this checkpoint; NEVER use future events."""
    if event_ids_by_checkpoint is None:
        return None
    value = event_ids_by_checkpoint.get((str(project), int(week), str(risk)), None)
    if value is None:
        return None
    if not isinstance(value, (set, list, tuple)):
        raise TypeError('Evidence IDs must be a list, set or tuple of strings')
    return frozenset(str(x) for x in value)


def replay_investigation_policy(predictions, policy=POLICY, event_ids_by_checkpoint=None):
    """Return (risk decisions, checkpoint decisions, graph-planner queue).

    Input contains one row per (project, checkpoint, risk), including frozen model
    probability, threshold and predicted label. No ground-truth labels are read.
    Separate state is maintained for each (project, risk). Replay sorts by UTC
    timestamp, then project and week, to avoid future-information leakage.
    """
    required = set(KEYS + ['risk', 'algorithm', 'probability', 'threshold', 'predicted_label'])
    if not required.issubset(predictions.columns):
        raise ValueError('Prediction columns missing: ' + str(sorted(required-set(predictions.columns))))
    if predictions.duplicated(KEYS + ['risk']).any():
        raise ValueError('Duplicate project/week/risk predictions')
    if not predictions['probability'].between(0, 1).all():
        raise ValueError('Probabilities outside [0,1]')
    if not np.isfinite(predictions['probability'].to_numpy(dtype=float)).all():
        raise ValueError('Nonfinite probabilities')
    if not np.array_equal(predictions['predicted_label'].to_numpy(dtype=int),
                          (predictions['probability'].to_numpy() >= predictions['threshold'].to_numpy()).astype(int)):
        raise ValueError('Frozen predicted labels disagree with frozen thresholds')
    work = predictions.copy()
    work['checkpoint_time'] = pd.to_datetime(work['checkpoint_time'], utc=True)
    work = work.sort_values(['checkpoint_time', 'instance_id', 'week_number', 'risk'], kind='stable')
    state = {}
    risk_decisions = []
    checkpoint_decisions = []
    queue = []
    for (project, week, timestamp), group in work.groupby(KEYS, sort=False):
        if len(group) != 3 or set(group['risk']) != set(TARGETS):
            raise ValueError(f'Expected three risk predictions at {project} W{week}')
        project = str(project); week = int(week); timestamp = _as_utc(timestamp)
        snapshot_id = f'{project}::W{week:02d}'
        selected = []
        all_risks = []
        for _, r in group.iterrows():
            risk = str(r['risk']); name = RISK_NAMES[risk]
            score = float(r['probability']); threshold = float(r['threshold'])
            above = bool(r['predicted_label'])
            key = (project, risk)
            previous = state.get(key, {'last_time': None, 'above': False,
                                        'last_investigation_time': None,
                                        'last_investigated_score': None,
                                        'last_evidence_ids': None})
            if previous['last_time'] is not None and timestamp <= previous['last_time']:
                raise ValueError(f'Non-increasing checkpoint time for {project} {risk}')
            current_ids = _safe_event_ids(event_ids_by_checkpoint, project, week, risk)
            new_evidence = (current_ids is not None and previous['last_evidence_ids'] is not None
                            and bool(current_ids - previous['last_evidence_ids']))
            last_inv_time = previous['last_investigation_time']
            weeks_since_inv = (float((timestamp-last_inv_time).total_seconds())/(7*86400)
                               if last_inv_time is not None else None)
            crossing = above and not previous['above']
            escalation = (above and previous['last_investigated_score'] is not None
                          and score-previous['last_investigated_score'] >= policy.escalation_delta
                          and weeks_since_inv is not None
                          and weeks_since_inv >= policy.minimum_escalation_gap_weeks)
            periodic = (above and weeks_since_inv is not None
                        and weeks_since_inv >= policy.periodic_refresh_weeks)
            evidence_changed = above and new_evidence
            if above and last_inv_time is None:
                reason = 'NEW_RISK'
            elif crossing:
                reason = 'NEW_RISK'
            elif evidence_changed:
                reason = 'NEW_EVIDENCE'
            elif escalation:
                reason = 'ESCALATION'
            elif periodic:
                reason = 'PERIODIC_REFRESH'
            elif above:
                reason = 'REUSE_MONITOR'
            else:
                reason = 'BELOW_THRESHOLD'
            schedule = reason in {'NEW_RISK','NEW_EVIDENCE','ESCALATION','PERIODIC_REFRESH'}
            info = {'risk_type': name, 'model': str(r['algorithm']),
                    'risk_score': score, 'threshold': threshold,
                    'predicted_risk': above,
                    'reliability': MODEL_SPEC[risk]['reliability'],
                    'shap_status': 'ON_DEMAND' if schedule else 'NOT_REQUESTED',
                    'top_shap_drivers': [], 'evidence_focus': EVIDENCE_HINTS[name],
                    'investigation_action': 'INVESTIGATE' if schedule else ('MONITOR_NO_NEW_REQUEST' if above else 'NO_RISK'),
                    'investigation_reason': reason,
                    'near_threshold': bool(abs(score-threshold) <= policy.near_threshold_margin),
                    'evidence_ids_available': current_ids is not None}
            all_risks.append(info)
            risk_decisions.append({'instance_id': project, 'week_number': week,
                'checkpoint_time': timestamp.isoformat(), 'risk': risk,
                'probability': score, 'threshold': threshold,
                'above_threshold': above, 'schedule_investigation': schedule,
                'action': info['investigation_action'], 'reason': reason,
                'new_evidence': bool(new_evidence),
                'weeks_since_investigation': weeks_since_inv})
            if schedule:
                selected.append(info)
                previous['last_investigation_time'] = timestamp
                previous['last_investigated_score'] = score
            previous['last_time'] = timestamp
            previous['above'] = above
            if current_ids is not None:
                previous['last_evidence_ids'] = current_ids
            state[key] = previous
        trigger_id = hashlib.sha256((snapshot_id+'|'+timestamp.isoformat()).encode()).hexdigest()[:20]
        checkpoint_decisions.append({'instance_id': project, 'week_number': week,
            'checkpoint_time': timestamp.isoformat(), 'snapshot_id': snapshot_id,
            'above_threshold_risks': sum(x['predicted_risk'] for x in all_risks),
            'investigation_scheduled': bool(selected),
            'scheduled_risks': len(selected),
            'reused_or_monitored_risks': sum(x['investigation_action']=='MONITOR_NO_NEW_REQUEST' for x in all_risks)})
        if selected:
            queue.append({'schema_version': '2.0', 'trigger_id': trigger_id,
                'snapshot_id': snapshot_id, 'instance_id': project, 'week_number': week,
                'checkpoint_time': timestamp.isoformat(), 'triggered': True,
                'trigger_rule': 'STATEFUL_NEW_ESCALATED_EVIDENCE_OR_PERIODIC',
                'requires_human_review': any(x['reliability']=='LOW_RELIABILITY' for x in selected),
                'risk_predictions': all_risks,
                'triggered_risks': [x['risk_type'] for x in selected],
                'investigation_reasons': {x['risk_type']:x['investigation_reason'] for x in selected},
                'graph_planner_input': {'project_id': project, 'as_of': timestamp.isoformat(),
                    'risk_types': [x['risk_type'] for x in selected],
                    'entity_focus': sorted(set(z for x in selected for z in x['evidence_focus'])),
                    'temporal_constraint': 'RETRIEVE_ONLY_EVIDENCE_AVAILABLE_AT_OR_BEFORE_CHECKPOINT'},
                'investigation_status': 'PENDING_ON_DEMAND_SHAP',
                'prediction_is_not_causal_evidence': True})
    risk_df = pd.DataFrame(risk_decisions)
    checkpoint_df = pd.DataFrame(checkpoint_decisions)
    if len(checkpoint_df) * 3 != len(risk_df):
        raise AssertionError('Risk/checkpoint coverage mismatch')
    return risk_df, checkpoint_df, queue


<div style="background:#edf4fc;border-left:6px solid #2865a0;padding:17px 20px;border-radius:10px"><span style="font-size:19px;font-weight:750;color:#17385a">12 · Replay existing predictions and measure savings</span><div style="font-size:13px;color:#43576d;margin-top:6px">Validation and previously inspected test predictions are replayed separately. The scheduler does not use labels or future project events.</div></div>

In [ ]:
validation_risk_decisions, validation_checkpoint_decisions, validation_queue = replay_investigation_policy(validation_predictions)
test_risk_decisions, test_checkpoint_decisions, test_queue = replay_investigation_policy(test_predictions)

def efficiency_summary(predictions, checkpoints, queue, partition):
    naive = int(predictions.groupby(KEYS)['predicted_label'].max().sum())
    scheduled = len(queue)
    return {'partition':partition, 'checkpoints':len(checkpoints),
        'naive_above_threshold_investigations':naive,
        'scheduled_investigations':scheduled,
        'investigations_avoided':naive-scheduled,
        'reduction_pct':100*(naive-scheduled)/naive if naive else 0,
        'scheduled_pct_of_checkpoints':100*scheduled/len(checkpoints)}

EFFICIENCY=pd.DataFrame([
    efficiency_summary(validation_predictions,validation_checkpoint_decisions,validation_queue,'validation'),
    efficiency_summary(test_predictions,test_checkpoint_decisions,test_queue,'exploratory_test')])
assert (EFFICIENCY.investigations_avoided>=0).all()
assert len(validation_risk_decisions)==198*3 and len(test_risk_decisions)==197*3
assert len(validation_queue)==int(validation_checkpoint_decisions.investigation_scheduled.sum())
assert len(test_queue)==int(test_checkpoint_decisions.investigation_scheduled.sum())
display(EFFICIENCY.round(2))
print('Risk-level action distribution:')
display(validation_risk_decisions.groupby(['risk','reason']).size().unstack(fill_value=0))
for name,rd,cd,queue in [
    ('validation',validation_risk_decisions,validation_checkpoint_decisions,validation_queue),
    ('exploratory_test',test_risk_decisions,test_checkpoint_decisions,test_queue)]:
    rd.to_csv(OUT/'triggers'/f'{name}_risk_decisions.csv',index=False)
    cd.to_csv(OUT/'triggers'/f'{name}_checkpoint_decisions.csv',index=False)
    with open(OUT/'triggers'/f'{name}_scheduled_investigations.jsonl','w',encoding='utf-8') as f:
        for item in queue:
            f.write(json.dumps(item,ensure_ascii=False,allow_nan=False)+'\n')
EFFICIENCY.to_csv(OUT/'metrics'/'investigation_efficiency.csv',index=False)


,partition,checkpoints,naive_above_threshold_investigations,scheduled_investigations,investigations_avoided,reduction_pct,scheduled_pct_of_checkpoints
0,validation,198,147,74,73,49.66,37.37
1,exploratory_test,197,151,65,86,56.95,32.99


Risk-level action distribution:


reason,BELOW_THRESHOLD,ESCALATION,NEW_RISK,PERIODIC_REFRESH,REUSE_MONITOR
risk,,,,,
dependency_risk_label,114,7,8,13,56
schedule_risk_label,91,0,24,13,70
scope_risk_label,133,5,16,5,39


<div style="background:#edf4fc;border-left:6px solid #2865a0;padding:17px 20px;border-radius:10px"><span style="font-size:19px;font-weight:750;color:#17385a">13 · On-demand SHAP and graph-planner readiness</span><div style="font-size:13px;color:#43576d;margin-top:6px">Only scheduled risks are explained. The bulk queue retains ON_DEMAND status; GraphRAG must enrich a queued payload before graph planning.</div></div>

In [ ]:
def enrich_scheduled_payload(payload, feature_row):
    """Explain only selected risk alerts. The returned payload is safe to hand to Graph Planner if READY."""
    enriched=json.loads(json.dumps(payload))
    for info in enriched['risk_predictions']:
        if info['investigation_action'] != 'INVESTIGATE':
            continue
        target=next(k for k,v in RISK_NAMES.items() if v==info['risk_type'])
        explanation=explain_snapshot(feature_row,target)
        delta=abs(float(info['risk_score'])-float(explanation['probability']))
        info.update({'top_shap_drivers':explanation['top_drivers'],
                     'shap_unit':explanation['shap_unit'],
                     'shap_inference_probability':float(explanation['probability']),
                     'inference_probability_difference':delta,
                     'shap_status':'COMPLETE' if delta<=1e-3 else 'RECHECK_INFERENCE'})
    selected=[x for x in enriched['risk_predictions'] if x['investigation_action']=='INVESTIGATE']
    enriched['investigation_status']=('READY_FOR_GRAPH_PLANNER' if selected and
        all(x['shap_status']=='COMPLETE' for x in selected) else 'RECHECK_INFERENCE')
    return enriched

# Demonstrate the complete handoff on one scheduled VALIDATION checkpoint.
if validation_queue:
    sample_payload=validation_queue[0]
    row=val[(val.instance_id==sample_payload['instance_id']) &
            (val.week_number==sample_payload['week_number'])]
    assert len(row)==1
    sample_enriched=enrich_scheduled_payload(sample_payload,row.iloc[0])
    (OUT/'triggers'/'sample_graph_planner_ready.json').write_text(
        json.dumps(sample_enriched,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
    print('Sample handoff status:',sample_enriched['investigation_status'])
    print('Scheduled risks:',sample_enriched['triggered_risks'])
    assert sample_enriched['investigation_status']=='READY_FOR_GRAPH_PLANNER', 'Inspect SHAP inference drift'
else:
    print('No validation risk scheduled; SHAP handoff example skipped.')


Sample handoff status: READY_FOR_GRAPH_PLANNER
Scheduled risks: ['schedule']


<div style="background:#edf4fc;border-left:6px solid #2865a0;padding:17px 20px;border-radius:10px"><span style="font-size:19px;font-weight:750;color:#17385a">14 · Live inference and persistent scheduling state</span><div style="font-size:13px;color:#43576d;margin-top:6px">For a new project, call the scorer on its point-in-time 34-feature vector, then the stateful manager. Keep the manager state between weeks.</div></div>

In [ ]:
def predict_current_risks(feature_row, instance_id, week_number, checkpoint_time):
    """Produce three model predictions; no ground-truth label required."""
    data=dict(feature_row)
    missing=[f for f in FEATURES if f not in data]
    if missing: raise ValueError(f'Missing required engineered features: {missing}')
    x=pd.DataFrame([{f:data[f] for f in FEATURES}])
    if not np.isfinite(x.to_numpy(dtype=float)).all():
        raise ValueError('Features must be finite numeric values')
    rows=[]
    for target in TARGETS:
        p=float(predict_positive(MODELS[target],x[MODEL_SPEC[target]['features']])[0])
        rows.append({'instance_id':str(instance_id),'week_number':int(week_number),
            'checkpoint_time':_as_utc(checkpoint_time),'risk':target,
            'algorithm':MODEL_SPEC[target]['algorithm'],'probability':p,
            'threshold':float(THRESHOLDS[target]),'predicted_label':int(p>=THRESHOLDS[target])})
    return pd.DataFrame(rows)

class StatefulInvestigationManager:
    """Append-only history replay: easy to audit and persist for a prototype.

    For real concurrent deployment, use an atomic transactional store instead.
    """
    def __init__(self, policy=POLICY):
        self.policy=policy
        self.prediction_history=pd.DataFrame()
        self.event_ids={}

    def process(self, prediction_frame, event_ids_by_risk=None):
        current=prediction_frame.copy()
        if len(current)!=3 or current['instance_id'].nunique()!=1 or current['week_number'].nunique()!=1:
            raise ValueError('Supply exactly three risks for one checkpoint')
        if not self.prediction_history.empty:
            current_keys=set(map(tuple,current[KEYS+['risk']].astype(str).to_numpy()))
            existing_keys=set(map(tuple,self.prediction_history[KEYS+['risk']].astype(str).to_numpy()))
            if current_keys & existing_keys:
                raise ValueError('Checkpoint already processed; use idempotent upstream request handling')
        proposed=pd.concat([self.prediction_history,current],ignore_index=True)
        new_ids=dict(self.event_ids)
        if event_ids_by_risk is not None:
            project=str(current.iloc[0]['instance_id']);week=int(current.iloc[0]['week_number'])
            for risk,ids in event_ids_by_risk.items():
                if risk not in TARGETS: raise ValueError('Unknown risk in evidence IDs')
                new_ids[(project,week,risk)]=list(ids)
        risk_decisions,checkpoint_decisions,queue=replay_investigation_policy(
            proposed,self.policy,event_ids_by_checkpoint=new_ids)
        # Commit only after validation succeeded
        self.prediction_history=proposed
        self.event_ids=new_ids
        project=str(current.iloc[0]['instance_id']);week=int(current.iloc[0]['week_number'])
        current_decisions=checkpoint_decisions[(checkpoint_decisions.instance_id==project)&
                                                (checkpoint_decisions.week_number==week)]
        assert len(current_decisions)==1
        matching=[q for q in queue if q['instance_id']==project and q['week_number']==week]
        return current_decisions.iloc[0].to_dict(), (matching[0] if matching else None)

    def save(self, folder):
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        if not self.prediction_history.empty:
            self.prediction_history.to_parquet(folder/'prediction_history.parquet',index=False)
        serialized=[{'project':p,'week':w,'risk':r,'ids':list(ids)}
                    for (p,w,r),ids in self.event_ids.items()]
        (folder/'event_ids.json').write_text(json.dumps(serialized,indent=2),encoding='utf-8')

    @classmethod
    def load(cls,folder,policy=POLICY):
        folder=Path(folder);obj=cls(policy)
        if (folder/'prediction_history.parquet').exists():
            obj.prediction_history=pd.read_parquet(folder/'prediction_history.parquet')
        if (folder/'event_ids.json').exists():
            for item in json.loads((folder/'event_ids.json').read_text(encoding='utf-8')):
                obj.event_ids[(item['project'],int(item['week']),item['risk'])]=item['ids']
        return obj

# Live-interface smoke test, no expensive SHAP inference.
_demo=StatefulInvestigationManager()
_demo_preds=predict_current_risks(val.iloc[0][FEATURES],val.iloc[0].instance_id,
                                 val.iloc[0].week_number,val.iloc[0].checkpoint_time)
_demo_decision,_demo_queue=_demo.process(_demo_preds)
assert isinstance(_demo_decision['investigation_scheduled'],(bool,np.bool_))
# Confirm persistence and chronological continuation without requiring a model refit.
_demo_dir = OUT / 'audits' / 'manager_state_smoke_test'
_demo.save(_demo_dir)
_restored_demo = StatefulInvestigationManager.load(_demo_dir)
assert len(_restored_demo.prediction_history)==3
assert set(_restored_demo.prediction_history.risk)==set(TARGETS)
print('PASS: fresh-checkpoint scorer, manager state persistence and reload')


PASS: fresh-checkpoint scorer, manager state persistence and reload


<div style="background:#edf4fc;border-left:6px solid #2865a0;padding:17px 20px;border-radius:10px"><span style="font-size:19px;font-weight:750;color:#17385a">15 · Final validation and Agentic GraphRAG contract</span><div style="font-size:13px;color:#43576d;margin-top:6px">Export independent model artifacts, policy, trigger decisions, on-demand SHAP entry points and a clear graph-planner schema.</div></div>

<div style="background:#e9f1fb;border-left:5px solid #2563eb;padding:14px;border-radius:8px"><b>14 · Final integrity checks, model restoration & Agentic GraphRAG handoff</b><div style="margin-top:6px">Validate the retained production pipeline, export its versioned contract and document the next-stage interface.</div></div>

In [ ]:
# Validate native CatBoost reload; verify TabPFN training context reconstructs the fitted context.
checks={}
for t in TARGETS:
    cols=MODEL_SPEC[t]['features']
    if MODEL_SPEC[t]['algorithm']=='CatBoost':
        restored=CatBoostClassifier()
        restored.load_model(str(OUT/'models'/f'{t}.cbm'))
        before=predict_positive(MODELS[t],val[cols].iloc[:5])
        after=predict_positive(restored,val[cols].iloc[:5])
        checks[f'{t}: model restored']=bool(np.allclose(before,after,atol=1e-9))
    else:
        context=pd.read_parquet(OUT/'models'/f'{t}_training_context.parquet')
        checks[f'{t}: training context restored']=bool(
            len(context)==len(train) and list(context.drop(columns=['target_label']).columns)==cols and
            np.array_equal(context.target_label.to_numpy(),train[t].to_numpy()))
checks['3 models fitted']=len(MODELS)==3
checks['3 validation thresholds']=len(THRESHOLDS)==3
checks['train/validation/test project disjoint']=not (set(train.instance_id)&set(val.instance_id) or set(train.instance_id)&set(test.instance_id) or set(val.instance_id)&set(test.instance_id))
checks['12 grouped CV folds trained']=len(cv_predictions)==592*3
checks['591 exploratory test predictions']=len(test_predictions)==197*3
checks['no duplicate test predictions']=not test_predictions.duplicated(KEYS+['risk']).any()
checks['SHAP explanations present']=len(shap_contributions)>0
checks['all SHAP contributions finite']=bool(np.isfinite(shap_contributions.shap_value).all())
checks['trigger decisions cover test checkpoints']=len(test_checkpoint_decisions)==197
checks['risk trigger queue matches decisions']=len(test_queue)==int(test_checkpoint_decisions.investigation_scheduled.sum())
checks['no future labels in handoff']=all('actual_label' not in json.dumps(x) for x in test_queue)
checks['no decision-risk model']='decision_risk_label' not in MODEL_SPEC
checks['scheduler never increases investigations']=bool((EFFICIENCY.investigations_avoided>=0).all())
checks['on-demand SHAP payload ready']=not validation_queue or sample_enriched['investigation_status']=='READY_FOR_GRAPH_PLANNER'
checks['all model probabilities finite']=bool(np.isfinite(test_predictions.probability).all())
validation_report=pd.DataFrame([{'validation':k,'status':'PASS' if v else 'FAIL'} for k,v in checks.items()])
display(validation_report)
validation_report.to_csv(OUT/'audits'/'final_validation_report.csv',index=False)
assert all(checks.values()),'Final validation failed; do not hand off artifacts'
print('ALL UNIFIED MODEL / SHAP / TRIGGER VALIDATIONS PASSED')

,validation,status
0,schedule_risk_label: training context restored,PASS
1,scope_risk_label: model restored,PASS
2,dependency_risk_label: model restored,PASS
3,3 models fitted,PASS
4,3 validation thresholds,PASS
5,train/validation/test project disjoint,PASS
6,12 grouped CV folds trained,PASS
7,591 exploratory test predictions,PASS
8,no duplicate test predictions,PASS
9,SHAP explanations present,PASS


ALL UNIFIED MODEL / SHAP / TRIGGER VALIDATIONS PASSED


In [ ]:
contract={
    'schema_version':'2.0',
    'architecture':'Temporal KG -> weekly features -> TabPFN/CatBoost -> investigation manager -> on-demand SHAP -> Agentic GraphRAG',
    'dataset_path':str(DATA_PATH), 'feature_list_path':str(FEATURE_PATH),
    'split_manifest_path':str(SPLIT_PATH),
    'evaluation_note':'Previously examined test projects; comparison is exploratory.',
    'risk_models':{risk:{'algorithm':MODEL_SPEC[risk]['algorithm'],
       'features':MODEL_SPEC[risk]['features'],
       'threshold':float(THRESHOLDS[risk]),
       'reliability':MODEL_SPEC[risk]['reliability'],
       'artifact':f'models/{risk}.cbm' if MODEL_SPEC[risk]['algorithm']=='CatBoost'
                  else f'models/{risk}_training_context.parquet'} for risk in TARGETS},
    'tabpfn_checkpoint':'v2.5',
    'tabpfn_n_estimators':TABPFN_ESTIMATORS,
    'catboost_hyperparameters':CAT_PARAMS,
    'investigation_policy':asdict(POLICY),
    'policy_priority':['NEW_RISK','NEW_EVIDENCE','ESCALATION','PERIODIC_REFRESH','REUSE_MONITOR','BELOW_THRESHOLD'],
    'evidence_note':'New evidence trigger requires real point-in-time event IDs supplied by caller. No IDs means no evidence-change trigger.',
    'shap_units':{'TabPFN-2.5':'probability','CatBoost':'log_odds'},
    'graph_planner_contract':{'input':'one schema_version 2.0 scheduled payload',
      'requires':'enrich_scheduled_payload() must yield READY_FOR_GRAPH_PLANNER',
      'temporal_rule':'retrieve only events observed by checkpoint time',
      'output':'evidence-backed event timeline and uncertainty-aware investigation report'}
}
(OUT/'efficient_investigation_contract.json').write_text(
    json.dumps(contract,indent=2,allow_nan=False),encoding='utf-8')
readme="""# Agentic GraphRAG handoff — Efficient Investigation Manager

**Risk models:** Schedule = TabPFN-2.5; Scope and Dependency = CatBoost. Inputs are the original 34 features from Notebook 2. Risk labels are never supplied at inference.

**Workflow:** score all three risks; compare to frozen validation thresholds; replay stateful investigation policy; run SHAP only for risks scheduled; pass READY_FOR_GRAPH_PLANNER payload to graph planner; retrieve/verify historical evidence; produce evidence-grounded report.

**Saved queues:** `triggers/validation_scheduled_investigations.jsonl` and `triggers/exploratory_test_scheduled_investigations.jsonl`. Queues are historical demos. A complete sample is `triggers/sample_graph_planner_ready.json`.

**Policy:** new risk crossing, material escalation, new source event IDs when explicitly supplied, or periodic refresh. Above-threshold unchanged risks are monitored without a new request. A previous report is reused only if the downstream investigator confirms it was completed; predictions are never deleted. Parameters in `efficient_investigation_contract.json` are operational heuristics, not performance-optimized thresholds.

**Event freshness:** event IDs must be truly available at or before `checkpoint_time`; numerical feature changes are not treated as source evidence. No event IDs supplied means only crossing, escalation and periodic refresh apply.

**SHAP:** `enrich_scheduled_payload(payload, feature_row)` fills top drivers only for scheduled risks. TabPFN uses permutation SHAP in probability units; CatBoost uses TreeSHAP in log-odds units. Do not compare SHAP magnitudes across model families or treat them as causal proof.

**Live interface:** `predict_current_risks(feature_row, instance_id, week_number, checkpoint_time)` then `StatefulInvestigationManager.process(prediction_frame, event_ids_by_risk=None)`. Save/load state between sessions; do not use a new empty manager every week.

**Graph planner:** use `graph_planner_input.project_id` and `graph_planner_input.as_of`, respect the strict as-of evidence constraint, verify source IDs and chronology, and report unsupported claims. Weak Scope/Dependency model alerts require review.

**Limitations:** cross-project generalization remains weak for Scope/Dependency; test projects were previously inspected; a lower investigation count does not improve prediction accuracy. New independent projects are required for unbiased evaluation of improvements.
"""
(OUT/'AGENTIC_GRAPHRAG_EFFICIENT_HANDOFF.md').write_text(readme,encoding='utf-8')
print('ALL EXPORTS READY:',OUT)
print('Handoff:',OUT/'AGENTIC_GRAPHRAG_EFFICIENT_HANDOFF.md')


ALL EXPORTS READY: /content/drive/MyDrive/GC1/Temporal_snapshots/model_training_efficient_investigation
Handoff: /content/drive/MyDrive/GC1/Temporal_snapshots/model_training_efficient_investigation/AGENTIC_GRAPHRAG_EFFICIENT_HANDOFF.md
